# GeoGuesser-RAG – Data Preparation and Embedding
This notebook prepares the cleaned country data, generates hint text, and computes embeddings for the GeoGuesser-RAG game. It uses the `countries_clean.csv` file created from the CIA Factbook dataset.


**Step 1:** Import the required libraries for data handling, embeddings, and vector search.


In [11]:


import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from IPython.display import Image, display



**Step 2:** Load the cleaned CSV file containing essential country data. Preview the first rows to confirm it looks correct.


In [12]:
# Load the countries dataset
df = pd.read_csv("raw/countries_clean.csv")
df.head()


,country,url,background,location,map_reference,area_comparative
0,Afghanistan,https://www.cia.gov/the-world-factbook/countri...,Ahmad Shah DURRANI unified the Pashtun tribes ...,"Southern Asia, north and west of Pakistan, eas...",Asia,almost six times the size of Virginia; slightl...
1,Akrotiri,https://www.cia.gov/the-world-factbook/countri...,NaN,NaN,NaN,NaN
2,Albania,https://www.cia.gov/the-world-factbook/countri...,After declaring independence from the Ottoman ...,"Southeastern Europe, bordering the Adriatic Se...",Europe,slightly smaller than Maryland
3,Algeria,https://www.cia.gov/the-world-factbook/countri...,"Algeria has known many empires and dynasties, ...","Northern Africa, bordering the Mediterranean S...",Africa,slightly less than 3.5 times the size of Texas
4,American Samoa,https://www.cia.gov/the-world-factbook/countri...,Tutuila -- the largest island in American Samo...,"Oceania, group of islands in the South Pacific...",Oceania,"slightly larger than Washington, DC"


**Step 3:** Build the main hint text for each country. For now, only the "fun fact" is used. You can later add more lines for richer hints or to support game modes like "Give me another hint."


In [13]:
# Combine relevant fields to create a single hint text per country
def make_hint(row):
    # Always start with the background as a "fun fact"
    hint = f"Fun Fact: {row['background']}"
    # Add other info if you want to enrich the hint step by step
    # hint += f"\nLocation: {row['location']}"
    # hint += f"\nMap Reference: {row['map_reference']}"
    # hint += f"\nArea Comparison: {row['area_comparative']}"
    return hint

df['hint_text'] = df.apply(make_hint, axis=1)
df[['country', 'hint_text']].head()


,country,hint_text
0,Afghanistan,Fun Fact: Ahmad Shah DURRANI unified the Pasht...
1,Akrotiri,Fun Fact: nan
2,Albania,Fun Fact: After declaring independence from th...
3,Algeria,Fun Fact: Algeria has known many empires and d...
4,American Samoa,Fun Fact: Tutuila -- the largest island in Ame...


**Step 4:** Use a pre-trained sentence transformer to generate vector embeddings for each hint. These vectors enable semantic search (retrieval) in the game.


In [14]:
# Load the sentence transformer model
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

# Generate embeddings for the hint texts
embeddings = model.encode(df['hint_text'].tolist(), show_progress_bar=True)


Batches: 100%|██████████| 9/9 [00:09<00:00,  1.10s/it]


**Step 5:** Create a FAISS index for fast similarity search based on hint embeddings. This is the retrieval engine for the GeoGuesser game.


In [15]:
# Create a FAISS index and add all country embeddings
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))


**Step 6:** Define a retrieval function. Given a user query, it returns the most relevant countries (and their hints) from the FAISS index.


In [16]:
def retrieve_country_hint(query, top_k=3):
    # Encode the query into an embedding
    query_embedding = model.encode([query])
    # Search for the most similar countries in the FAISS index
    D, I = index.search(np.array(query_embedding), top_k)
    # Return the top-k countries and their hints
    return df.iloc[I[0]][['country', 'hint_text']]

    # Example usage
retrieve_country_hint("island in the Caribbean", top_k=3)



,country,hint_text
238,Turks and Caicos Islands,Fun Fact: The islands were part of the UK's Ja...
17,"Bahamas, The",Fun Fact: Lucayan Indians inhabited the Bahama...
168,Nicaragua,Fun Fact: The Pacific coast of Nicaragua was s...


## Game Loop: Guess the Country
In this basic version, the notebook randomly selects a country and presents its hints. The user must type their guess, and the code checks if it is correct.


**Game Logic:**
- The function randomly selects a country from the dataset for each round.
- The user is presented with a hint and types their guess.
- The answer is checked for correctness (case insensitive).
- The game tracks the score and displays it at the end.


In [17]:
import gradio as gr

# Define game modes and which fields to use for hints
HINT_FIELDS = {
    "Guess by Fun Fact": ["background"],
    "Guess by Location": ["location"],
    "Guess by Area Comparison": ["area_comparative"],
    "Mixed Mode": ["background", "location", "area_comparative"]
}

def new_question(game_mode):
    idx = np.random.randint(len(df))
    row = df.iloc[idx]
    # Prepare hints based on selected mode
    hints = []
    for field in HINT_FIELDS[game_mode]:
        value = row[field]
        if pd.notnull(value) and value.strip():
            hints.append(f"{field.replace('_', ' ').title()}: {value}")
    # First hint only; user can ask for more later
    hint_history = [hints[0]] if hints else ["No data available."]
    return [idx, hint_history, False, game_mode], hint_history[0], "", "Start guessing!"

def request_hint(state):
    idx, hint_history, guessed, game_mode = state
    row = df.iloc[idx]
    hints = []
    for field in HINT_FIELDS[game_mode]:
        value = row[field]
        if pd.notnull(value) and value.strip():
            hints.append(f"{field.replace('_', ' ').title()}: {value}")
    # Reveal the next unrevealed hint
    if len(hint_history) < len(hints):
        hint_history.append(hints[len(hint_history)])
    return [idx, hint_history, guessed, game_mode], "\n".join(hint_history), "", "Here's another hint!"

def submit_guess(user_guess, state):
    idx, hint_history, guessed, game_mode = state
    country = df.iloc[idx]['country']
    if guessed:
        feedback = "You already guessed! Start a new game."
        return state, "\n".join(hint_history), "", feedback
    if user_guess.strip().lower() == country.strip().lower():
        feedback = f"✅ Correct! It was {country}."
        guessed = True
    else:
        feedback = f"❌ Wrong. The correct answer was: {country}."
        guessed = True
    return [idx, hint_history, guessed, game_mode], "\n".join(hint_history), "", feedback

with gr.Blocks() as demo:
    gr.Markdown("# 🌍 GeoGuesser-RAG (CIA Factbook Edition)")
    state = gr.State()
    game_mode = gr.Radio(
        list(HINT_FIELDS.keys()), 
        label="Choose Game Mode", 
        value="Guess by Fun Fact"
    )
    with gr.Row():
        with gr.Column():
            hint_box = gr.Markdown()
            hint_btn = gr.Button("Get another hint")
        with gr.Column():
            input_box = gr.Textbox(label="Your guess", placeholder="Enter country name")
            feedback_box = gr.Markdown()
    guess_btn = gr.Button("Submit Guess")
    start_btn = gr.Button("Start New Game")

    # When the game mode changes or a new game starts
    game_mode.change(fn=new_question, inputs=[game_mode], outputs=[state, hint_box, input_box, feedback_box])
    start_btn.click(fn=new_question, inputs=[game_mode], outputs=[state, hint_box, input_box, feedback_box])
    # Get another hint
    hint_btn.click(fn=request_hint, inputs=[state], outputs=[state, hint_box, input_box, feedback_box])
    # Submit a guess
    guess_btn.click(fn=submit_guess, inputs=[input_box, state], outputs=[state, hint_box, input_box, feedback_box])

demo.launch()


* Running on local URL:  http://127.0.0.1:7866

To create a public link, set `share=True` in `launch()`.
